In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
# import xgboost as xgb
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from ax.generation_strategy.center_generation_node import CenterGenerationNode
from ax.generation_strategy.transition_criterion import MinTrials
from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationNode
from ax.generation_strategy.model_spec import GeneratorSpec
from ax.modelbridge.registry import Generators
from gpytorch.kernels import MaternKernel
from botorch.models import SingleTaskGP
from botorch.models.transforms.input import Warp
from botorch.models.map_saas import AdditiveMapSaasSingleTaskGP
from ax.utils.stats.model_fit_stats import MSE
from ax.models.torch.botorch_modular.surrogate import SurrogateSpec, ModelConfig
from botorch.acquisition.logei import qLogNoisyExpectedImprovement
import seaborn

In [2]:
o = 27
n = 14
s = o-n
sampler = Sampler_class()
Parameters_lis = [
    RangeParameterConfig(name="s1", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="s2", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="b1", parameter_type="float", bounds=tuple([0,1])),
    ]

In [3]:
client = Client()
gp_model = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/Modelling/ModelMk4.json")
gp_model.get_next_trials(max_trials=1)
def SurrogateModelOfReality(s1, s2, b1):
    y_pred = gp_model.predict([{"s1":s1,"s2":s2,"b1":b1}])[0]["t1"][0]
    return np.float64(y_pred)

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


In [4]:
y_max_lis = []

for i in range(100):
    client = Client()
    parameters = [
        RangeParameterConfig(
            name="s1", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="s2", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="b1", parameter_type="float", bounds=(0, 1)
        ),
    ]
    client.configure_experiment(parameters=parameters)
    def construct_generation_strategy(
        generator_spec: GeneratorSpec, node_name: str,
    ) -> GenerationStrategy:
        """Constructs a Center + Sobol + Modular BoTorch `GenerationStrategy`
        using the provided `generator_spec` for the Modular BoTorch node.
        """
        botorch_node = GenerationNode(
            node_name=node_name,
            model_specs=[generator_spec],
        )
        return GenerationStrategy(
            name=f"{node_name}",
            nodes=[botorch_node]
        )

    # Let's construct the simplest version with all defaults.
    construct_generation_strategy(
        generator_spec=GeneratorSpec(model_enum=Generators.BOTORCH_MODULAR),
        node_name="Modular BoTorch",
    )

    surrogate_spec = SurrogateSpec(
        model_configs=[
            # Select between two models:
            # An additive mixture of relatively strong SAAS priors with input Warping.
            # A relatively vanilla GP with a Matern kernel.
            ModelConfig(
                botorch_model_class=SingleTaskGP,
                covar_module_class=MaternKernel,
                covar_module_options={"nu": 2.5},
            ),
        ],
        eval_criterion=MSE,  # Select the model to use as the one that minimizes mean squared error.
        allow_batched_models=False,  # Forces each metric to be modeled with an independent BoTorch model.
        # If we wanted to specify different options for different metrics.
        # metric_to_model_configs: dict[str, list[ModelConfig]]
    )

    generator_spec = GeneratorSpec(
        model_enum=Generators.BOTORCH_MODULAR,
        model_kwargs={
            "surrogate_spec": surrogate_spec,
            "botorch_acqf_class": qLogNoisyExpectedImprovement,
            # Can be used for additional inputs that are not constructed
            # by default in Ax. We will demonstrate below.
            "acquisition_options": {},
        },
        # We can specify various options for the optimizer here.
        model_gen_kwargs = {
            "model_gen_options": {
                "optimizer_kwargs": {
                    "num_restarts": 20,
                    "sequential": False,
                    "options": {
                        "batch_limit": 5,
                        "maxiter": 200,
                    },
                },
            },
        }
    )

    generation_strategy = construct_generation_strategy(
        generator_spec=generator_spec,
        node_name="BoTorch w/ Model Selection",
    )
    generation_strategy

    client.set_generation_strategy(
        generation_strategy=generation_strategy,
    )

    metric_name = "t1" # this name is used during the optimization loop in Step 5
    objective = f"{metric_name}" # minimization is specified by the negative sign

    client.configure_optimization(objective=objective)

    X = sampler.three.PseudorandomSampler3D_func(n,Parameters_lis).T

    for array in X:
        my_parameters = {"s1": array[0], "s2": array[1], "b1": array[2]}
        trial_index = client.attach_trial(parameters=my_parameters)
        client.complete_trial(trial_index=trial_index,raw_data={"t1": SurrogateModelOfReality(**my_parameters)})

    for _ in range(s): # Run 10 rounds of trials
        # We will request three trials at a time in this example
        trials = client.get_next_trials(max_trials=1)

        for trial_index, parameters in trials.items():
            s1 = parameters["s1"]
            s2 = parameters["s2"]
            b1 = parameters["b1"]

            result = SurrogateModelOfReality(s1, s2, b1)

            # Set raw_data as a dictionary with metric names as keys and results as values
            raw_data = {metric_name: result}

            # Complete the trial with the result
            client.complete_trial(trial_index=trial_index, raw_data=raw_data)
    # print(client.summarize())
    print(f"Trial {i} =========================================")
    y_max = np.max(np.array(client.summarize().t1))
    print(y_max)
    y_max_lis.append(y_max)
    print()

y_max_arr = np.array(y_max_lis)
print(y_max_arr)

Trial 0 =========================================
14.277060883083108



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 1 =========================================
16.243192355396253

Trial 2 =========================================
14.369892642321258

Trial 3 =========================================
15.85827086480607



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 4 =========================================
16.481129047143945

Trial 5 =========================================
13.514065628717148

Trial 6 =========================================
15.392502002813071

Trial 7 =========================================
13.847184186826972

Trial 8 =========================================
18.587448730728138

Trial 9 =========================================
15.196350111138168

Trial 10 =========================================
13.416242876119114

Trial 11 =========================================
15.874662610597563



...

Trial 68 =========================================
16.2386308648688

Trial 69 =========================================
18.788161991949753

Trial 70 =========================================
18.795931018624117

Trial 71 =========================================
14.032047081800473

Trial 72 =========================================
15.295599361028577

Trial 73 =========================================
15.805770478532741

Trial 74 =========================================
14.262104047737852

Trial 75 =========================================
15.880946703302996

Trial 76 =========================================
15.868255881032427



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 77 =========================================
16.110703638158494

Trial 78 =========================================
15.393249323774116

Trial 79 =========================================
15.363496417435334

Trial 80 =========================================
16.187905297483347

Trial 81 =========================================
17.25191979131571

Trial 82 =========================================
17.587684502298643

Trial 83 =========================================
13.863225467515994

Trial 84 =========================================
14.966954345935049

Trial 85 =========================================
16.180575355584647

Trial 86 =========================================
15.836908063240823

Trial 87 =========================================
15.703301629430442

Trial 88 =========================================
15.343131160835307



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/botorch/optim/optimize.py:331: BadInitialCandidatesWarning: Unable to find non-zero acquisition function values - initial conditions are being selected randomly.
  generated_initial_conditions = opt_inputs.get_ic_generator()(


Trial 89 =========================================
18.817135951322122

Trial 90 =========================================
14.27838825018819

Trial 91 =========================================
14.420328111692342

Trial 92 =========================================
13.599176875568105

Trial 93 =========================================
15.333785945469812

Trial 94 =========================================
14.147423634277635

Trial 95 =========================================
14.269376651203771

Trial 96 =========================================
16.240634717214558

Trial 97 =========================================
15.881126631393844

Trial 98 =========================================
15.18061130668052

Trial 99 =========================================
17.510050137389744

[14.27706088 16.24319236 14.36989264 15.85827086 16.48112905 13.51406563
 15.392502   13.84718419 18.58744873 15.19635011 13.41624288 15.87466261
 16.20267102 15.36902542 14.73499785 13.87515206 14.26359058 13.78794147
 1

In [5]:
print(f"Max = {np.max(y_max_arr)}")
print(f"Avg = {np.average(y_max_arr)}")
print(f"Std = {np.std(y_max_arr)}")

Max = 18.817135951322122
Avg = 15.399018784654043
Std = 1.2750818960532397


In [6]:
print(y_max_arr.tolist())

[14.277060883083108, 16.243192355396253, 14.369892642321258, 15.85827086480607, 16.481129047143945, 13.514065628717148, 15.392502002813071, 13.847184186826972, 18.587448730728138, 15.196350111138168, 13.416242876119114, 15.874662610597563, 16.20267102136394, 15.369025419492576, 14.73499785045305, 13.875152061276875, 14.263590578656817, 13.787941471305606, 15.377996552318244, 15.053350696446868, 14.850669786420728, 15.197101075991533, 14.441107377700835, 14.929770448992905, 14.268759491873807, 15.929222010399386, 16.033903296151077, 14.113080393289787, 15.849319029224368, 16.404875155408792, 18.734934526283922, 16.29828996648869, 14.373299157202371, 14.47256882362769, 14.266523966788382, 16.136941615448332, 17.55750694613745, 15.392058182748034, 14.595023859876529, 15.343957179633382, 16.437607458322816, 14.26926407828881, 14.290423274900373, 14.259185185375298, 15.74228589954276, 14.350139764494267, 14.501912566066474, 15.35120334059897, 14.512741623295103, 15.962577113493506, 16.22358

In [7]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/SequentialTestingOfSamplingTechnique/DataGenerated/BOpt-EI-9,27,1-14.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
latestdf = pd.DataFrame(y_max_arr)
newdf = pd.concat(objs=[loadeddf,latestdf],axis=0)
newdf = newdf.reset_index(drop=True)
pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)